# Headless Cellpose SAM segmentation

This notebook runs a [Fractal](https://fractal-analytics-platform.github.io/) task from Python, without the napari plugin's widgets, using the API of `workflow_tasks_core`:

- `Catalog`: the Fractal task-package catalog,
- `Package`: one version of a task package, installed into its own pixi environment,
- `Task`: one task of a package: its arguments, running it and finding its outputs.

It segments nuclei in an OME-Zarr image with the **Cellpose SAM** task, then opens the result in napari. Everything up to the last section works without napari, e.g. on a headless machine.

**Requirements**
- `pixi` on your `PATH` (or in `~/.pixi/bin`). Each task package runs in its own pixi environment.
- A kernel with `napari-workflow-tasks` installed, e.g. the repository's pixi environment (`pixi run python -m ipykernel install --user --name napari-workflow-tasks`).
- The example dataset: the MIP of `20200812-CardiomyocyteDifferentiation14-Cycle1` from [Zenodo](https://zenodo.org/records/10424292), unzipped.

## Settings

In [ ]:
from pathlib import Path

# The OME-Zarr plate (the task writes its label image into it)
PLATE = Path.home() / "Downloads" / "20200812-CardiomyocyteDifferentiation14-Cycle1_mip.zarr"
zarr_url = str(PLATE / "B" / "03" / "0")  # well B03, image 0
zarr_url

Fractal tasks write their results (label images, tables) into the OME-Zarr they run on, next to the existing data: nothing is copied. Here, the segmentation adds a label image `DAPI_cellpose_sam` to the image (`label_name` below), replacing one of that name from an earlier run.

## Get the task package

`Catalog().package()` finds a package by name in the Fractal task catalog. Its newest stable version is used unless you pass `version=`.

`Catalog().search("cellpose")` lists matching tasks of all packages. It reads the whole catalog, which can take a while, so it is not run here.

In [ ]:
from workflow_tasks_core import Catalog

pkg = Catalog().package("fractal-cellpose-sam-task")
print(pkg)
print("Versions:", pkg.versions)
print("Tasks:", pkg.tasks)

## Install it

`install()` creates a pixi environment for this package version under `~/.napari_workflow_tasks/envs/` and checks that its tasks import. The first install downloads the dependencies (PyTorch, Cellpose, ...) and takes several minutes. Once installed, it takes about a second.

Later, `Package.installed("fractal-cellpose-sam-task")` gets the installed package without the catalog, also offline.

In [ ]:
pkg.install()

## Look at the task's arguments

`help()` lists the arguments: type, default and description, required ones first. Fields of argument models (such as `channels`) are indented below them. Arguments that `run()` fills in (`zarr_url`, ...) are not listed.

In [ ]:
seg = pkg.task("Cellpose SAM Segmentation")
seg.help()

`validate()` checks arguments before anything runs, and `run()` does so too. Mistakes are reported all at once:

In [ ]:
from workflow_tasks_core import SchemaError

try:
    seg.validate({"channel": {"identifiers": ["DAPI"]}, "overwrite": "yes"})
except SchemaError as e:
    print(e)

## Run it

`run()` takes an image (or a list of images, or a plate) and the task's arguments as keyword arguments. Arguments left out use the task's defaults. It runs the task in the package's pixi environment, prints its log and returns when it is done. `log=None` hides the log.

Here, the nuclei in the DAPI channel are segmented in each field of view (`FOV_ROI_table`). Leave out `iterator_configuration` to segment the whole image at once.

In [ ]:
seg.run(
    zarr_url,
    channels={"identifiers": ["DAPI"]},
    label_name="{channel_identifier}_cellpose_sam",  # -> DAPI_cellpose_sam
    iterator_configuration={"roi_table": "FOV_ROI_table"},
)

## Find its outputs

The results are in the OME-Zarr. `outputs()` gives the paths of the label images, tables and new images the run wrote. `missing` lists expected outputs that are not on disk.

In [ ]:
print(seg.outputs())
print("Missing:", seg.missing)

To work with the data, open the image with [ngio](https://biovisioncenter.github.io/ngio/):

In [ ]:
import numpy as np
from ngio import open_ome_zarr_container

label_name = seg.outputs("labels")[0].rsplit("/", 1)[-1]
nuclei = open_ome_zarr_container(zarr_url).get_label(label_name).get_array()
print(f"{label_name}: {len(np.unique(nuclei)) - 1} nuclei")

To run another task on the results, pass `seg.zarr_urls`: the images to continue with, including any new images a task created.

## Show it in napari

`show()` opens the input image with the label images this run wrote (not the image's other labels) in napari.

In [ ]:
viewer = seg.show()

In [ ]:
from napari.utils import nbscreenshot

viewer.reset_view()
nbscreenshot(viewer)